In [ ]:

import pandas as pd
from pathlib import Path

project_dir = Path.cwd().parent
raw_dir = project_dir / "data" / "raw"

df = pd.read_csv(
    raw_dir / "da_agriculture_yields.csv"
)

print("DATASET SHAPE")
print(df.shape)

print("\nCOLUMN NAMES")
print(df.columns.tolist())

print("\nFIRST 5 ROWS")
print(df.head().to_string(index=False))

print("\nMISSING VALUES")
print(df.isna().sum())

print("\nDATA TYPES")
print(df.dtypes)

# Identify potential crop, year and weather columns
for col in df.columns:
    if any(
        word in col.lower()
        for word in ["crop", "year", "state", "rain", "temp", "soil"]
    ):
        print(f"\n{col}:")
        if df[col].nunique() <= 30:
            print(df[col].value_counts(dropna=False).head(30))
        else:
            print(df[col].describe())


In [ ]:

import pandas as pd
from pathlib import Path

raw_dir = Path.cwd().parent / "data" / "raw"

df = pd.read_csv(raw_dir / "da_agriculture_yields.csv")

target_crops = [
    "Rice", "Soybean", "Cotton", "Sugarcane", "Tomato"
]

df = df[df["crop"].isin(target_crops)].copy()

# Check duplicate geographical and yearly records
keys = ["state", "district", "crop", "year"]

print("TARGET CROP RECORDS:", len(df))
print("DUPLICATE STATE-DISTRICT-CROP-YEAR KEYS:",
      df.duplicated(keys).sum())

# Count valid consecutive-year pairs
unique_records = df[keys].drop_duplicates()

previous_year = unique_records.copy()
previous_year["year"] += 1

matched = unique_records.merge(
    previous_year,
    on=keys,
    how="inner"
)

print("\nVALID PREVIOUS-YEAR PAIRS BY CROP")
print(
    matched["crop"]
    .value_counts()
    .reindex(target_crops, fill_value=0)
)

print("\nRECORDS BY CROP AND YEAR")
print(
    pd.crosstab(df["crop"], df["year"])
    .reindex(target_crops)
)

print("\nRECORDS BY STATE AND TARGET CROP")
print(
    pd.crosstab(df["state"], df["crop"])
    .reindex(columns=target_crops, fill_value=0)
)


In [ ]:

%pip install -q scikit-learn joblib matplotlib


In [ ]:

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

project_dir = Path.cwd().parent
raw_dir = project_dir / "data" / "raw"
models_dir = project_dir / "models" / "week3"
reports_dir = project_dir / "reports"
figures_dir = reports_dir / "figures"

models_dir.mkdir(parents=True, exist_ok=True)
figures_dir.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(raw_dir / "da_agriculture_yields.csv")

crops = ["Rice", "Soybean", "Cotton", "Sugarcane", "Tomato"]

numeric_features = [
    "rainfall_mm",
    "area_hectares",
    "fertiliser_kg_ha",
    "pesticide_used"
]

categorical_features = [
    "state",
    "season",
    "irrigation"
]

features = numeric_features + categorical_features
target = "yield_tonnes_per_ha"

results = []

for crop in crops:
    crop_df = df[df["crop"] == crop].copy()

    # Train on earlier years, test on 2024-2025
    train = crop_df[crop_df["year"] <= 2023]
    test = crop_df[crop_df["year"] >= 2024]

    if len(train) < 10 or len(test) < 5:
        print(f"Skipping {crop}: insufficient records")
        continue

    preprocessing = ColumnTransformer([
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical_features
        )
    ])

    model = Pipeline([
        ("preprocessing", preprocessing),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=200,
                random_state=42,
                min_samples_leaf=2
            )
        )
    ])

    model.fit(train[features], train[target])

    predictions = model.predict(test[features])
    actual = test[target].to_numpy()

    rmse = np.sqrt(mean_squared_error(actual, predictions))

    # MAPE only for nonzero actual yields
    nonzero = actual != 0
    mape = (
        np.mean(
            np.abs(
                (actual[nonzero] - predictions[nonzero])
                / actual[nonzero]
            )
        ) * 100
        if nonzero.any()
        else np.nan
    )

    results.append({
        "crop": crop,
        "train_records": len(train),
        "test_records": len(test),
        "RMSE": round(rmse, 4),
        "MAPE_percent": round(mape, 2)
    })

    joblib.dump(
        model,
        models_dir / f"{crop.lower()}_model.joblib"
    )

    plt.figure(figsize=(6, 5))
    plt.scatter(actual, predictions, alpha=0.75)

    low = min(actual.min(), predictions.min())
    high = max(actual.max(), predictions.max())

    plt.plot(
        [low, high],
        [low, high],
        linestyle="--",
        color="gray"
    )

    plt.xlabel("Actual Yield (tonnes/ha)")
    plt.ylabel("Predicted Yield (tonnes/ha)")
    plt.title(f"{crop}: Actual vs Predicted Yield")
    plt.tight_layout()

    plt.savefig(
        figures_dir / f"week3_{crop.lower()}_predictions.png",
        dpi=300
    )
    plt.show()

    print(
        f"{crop}: RMSE={rmse:.4f}, "
        f"MAPE={mape:.2f}%"
    )

results_df = pd.DataFrame(results)

results_df.to_csv(
    reports_dir / "week3_model_metrics.csv",
    index=False
)

print("\nFINAL MODEL COMPARISON")
print(results_df.to_string(index=False))


In [ ]:

import pandas as pd
import numpy as np
import joblib

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

project_dir = Path.cwd().parent
df = pd.read_csv(
    project_dir / "data" / "raw" / "da_agriculture_yields.csv"
)

cotton = df[df["crop"] == "Cotton"].copy()

train = cotton[cotton["year"] <= 2023]
test = cotton[cotton["year"] >= 2024]

base_numeric = [
    "rainfall_mm",
    "area_hectares",
    "fertiliser_kg_ha",
    "pesticide_used"
]

categorical = [
    "state",
    "season",
    "irrigation"
]

def train_cotton_model(numeric_features):
    features = numeric_features + categorical

    preprocessor = ColumnTransformer([
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric_features
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            categorical
        )
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            min_samples_leaf=2
        ))
    ])

    model.fit(train[features], train["yield_tonnes_per_ha"])

    actual = test["yield_tonnes_per_ha"].to_numpy()
    predicted = model.predict(test[features])

    rmse = np.sqrt(mean_squared_error(actual, predicted))

    valid = actual != 0
    mape = (
        np.mean(
            np.abs(
                (actual[valid] - predicted[valid]) / actual[valid]
            )
        ) * 100
        if valid.any() else np.nan
    )

    return model, rmse, mape

baseline_model, baseline_rmse, baseline_mape = (
    train_cotton_model(base_numeric)
)

enhanced_model, enhanced_rmse, enhanced_mape = (
    train_cotton_model(base_numeric + ["year"])
)

comparison = pd.DataFrame([
    {
        "model": "Baseline Cotton",
        "RMSE": baseline_rmse,
        "MAPE_percent": baseline_mape
    },
    {
        "model": "Cotton + Year",
        "RMSE": enhanced_rmse,
        "MAPE_percent": enhanced_mape
    }
])

print("COTTON MODEL COMPARISON")
print(comparison.round(4).to_string(index=False))

comparison.to_csv(
    project_dir / "reports" / "week3_cotton_comparison.csv",
    index=False
)

# Save enhanced model only if RMSE improves
if enhanced_rmse < baseline_rmse:
    joblib.dump(
        enhanced_model,
        project_dir / "models" / "week3" / "cotton_enhanced_model.joblib"
    )
    print("\nEnhanced Cotton model improved RMSE.")
else:
    print("\nNo RMSE improvement from adding year.")


In [ ]:

from pathlib import Path
import pandas as pd

project_dir = Path.cwd().parent
reports_dir = project_dir / "reports"

metrics = pd.read_csv(
    reports_dir / "week3_model_metrics.csv"
)

cotton = pd.read_csv(
    reports_dir / "week3_cotton_comparison.csv"
)

def markdown_table(df):
    headers = list(df.columns)
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |"
    ]
    for row in df.itertuples(index=False, name=None):
        lines.append(
            "| " + " | ".join(str(v) for v in row) + " |"
        )
    return "\n".join(lines)

report = f"""# KisanAI AgriTech — Week 3 Model Training Report

## Objective
Train separate crop-yield prediction models and evaluate
their performance using RMSE and MAPE.

## Dataset
Source: Client-provided da_agriculture_yields.csv

- Total dataset: 600 records
- Target-crop records: 288
- Crops: Rice, Soybean, Cotton, Sugarcane, Tomato
- Years: 2020–2025
- Training years: 2020–2023
- Testing years: 2024–2025

## Methodology
- Algorithm: Random Forest Regressor
- One independent model per crop
- Numeric features: rainfall, area, fertiliser, pesticide use
- Categorical features: state, season, irrigation
- Missing-value imputation and one-hot encoding
- Chronological train/test split

## Model Evaluation

{markdown_table(metrics.round(4))}

RMSE is reported in tonnes per hectare.
MAPE is reported as a percentage.

## Cotton Feature Experiment

{markdown_table(cotton.round(4))}

Adding year reduced Cotton RMSE but slightly increased MAPE.
This experiment was evaluated on the existing test set,
so it is exploratory rather than a final unbiased
model-selection result.

## Key Findings
- Sugarcane achieved the lowest RMSE and MAPE.
- Rice showed the highest MAPE.
- Cotton was not the worst-performing crop.
- All models have substantial prediction errors.

## Limitations
- Small training datasets for each crop.
- Dataset does not include Telangana.
- District names are generic identifiers.
- Rainfall is the only available weather variable.
- Valid previous-year yield pairs are insufficient
  for reliable lag-feature modelling.
- Soil-type data was not available.
- No external government-weather validation was performed.
- Test-set reuse in the Cotton experiment may bias
  any selected-model performance estimate.

## Conclusion
Five exploratory crop-specific Random Forest models
were trained and evaluated using RMSE and MAPE.
The models provide a baseline for further feature
engineering, data integration and validation.

The client's full weather-plus-lag-feature requirement
remains partially unmet.
"""

output = reports_dir / "week3_model_report.md"
output.write_text(report, encoding="utf-8")

print("Week 3 report saved successfully!")
print(output)


In [ ]:

from pathlib import Path

project_dir = Path.cwd().parent

required = [
    "notebooks/02_crop_yield_models.ipynb",
    "reports/week3_model_report.md",
    "reports/week3_model_metrics.csv",
    "reports/week3_cotton_comparison.csv",
    "models/week3/rice_model.joblib",
    "models/week3/soybean_model.joblib",
    "models/week3/cotton_model.joblib",
    "models/week3/sugarcane_model.joblib",
    "models/week3/tomato_model.joblib",
]

for name in required:
    path = project_dir / name
    print("FOUND" if path.exists() else "MISSING", "-", name)


In [ ]:

from pathlib import Path

project_dir = Path.cwd().parent

files = [
    "notebooks/02_crop_yield_models.ipynb",
    "reports/week3_model_report.md",
    "reports/week3_model_metrics.csv",
    "reports/week3_cotton_comparison.csv",
    "models/week3/rice_model.joblib",
    "models/week3/soybean_model.joblib",
    "models/week3/cotton_model.joblib",
    "models/week3/sugarcane_model.joblib",
    "models/week3/tomato_model.joblib"
]

print("WEEK 3 FILE VERIFICATION")
print("-" * 50)

for filename in files:
    path = project_dir / filename
    print(
        "FOUND  " if path.exists() else "MISSING",
        filename
    )


In [ ]:

import pandas as pd
from pathlib import Path

project_dir = Path.cwd().parent

file_path = (
    project_dir / "data" / "external"
    / "Crop_recommendation.csv"
)

df = pd.read_csv(file_path)

print("DATASET SHAPE:", df.shape)

print("\nCOLUMN NAMES:")
print(df.columns.tolist())

print("\nFIRST FIVE ROWS:")
print(df.head().to_string(index=False))

print("\nMISSING VALUES:")
print(df.isnull().sum())

print("\nCROP COUNTS:")
print(df["label"].value_counts())


In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
from pathlib import Path

features = [
    "N", "P", "K",
    "temperature", "humidity",
    "ph", "rainfall"
]

X = df[features]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    min_samples_leaf=2
)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("CROP RECOMMENDATION MODEL")
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
print(f"Accuracy: {accuracy * 100:.2f}%")

print("\nCLASSIFICATION REPORT")
print(classification_report(y_test, predictions))

# Save model
project_dir = Path.cwd().parent
model_dir = project_dir / "models" / "soil_weather"
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    model,
    model_dir / "crop_recommendation_model.joblib"
)

print("\nModel saved successfully!")


In [ ]:

from pathlib import Path
from sklearn.metrics import classification_report, confusion_matrix
import pandas as pd

project_dir = Path.cwd().parent
report_dir = project_dir / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

report_df = pd.DataFrame(
    classification_report(
        y_test,
        predictions,
        output_dict=True
    )
).transpose()

report_df.to_csv(
    report_dir / "soil_weather_classification_report.csv"
)

confusion_df = pd.DataFrame(
    confusion_matrix(y_test, predictions, labels=model.classes_),
    index=model.classes_,
    columns=model.classes_
)

confusion_df.to_csv(
    report_dir / "soil_weather_confusion_matrix.csv"
)

summary = """# KisanAI Soil & Weather Crop Recommendation

## Dataset
Kaggle Crop Recommendation Dataset

- Records: 2,200
- Crop classes: 22
- Training records: 1,760
- Testing records: 440

## Model
Random Forest Classifier

## Input Features
Nitrogen, phosphorus, potassium, temperature,
humidity, soil pH and rainfall.

## Evaluation
Held-out test accuracy: 99.32%

## Limitations
This is a crop recommendation classifier, not a
crop-yield regression model.

The dataset does not contain actual yield values,
soil-type categories, district identifiers or years.

Results have not been validated on independent
real-world agricultural data.
"""

(report_dir / "soil_weather_model_report.md").write_text(
    summary,
    encoding="utf-8"
)

print("Recommendation reports saved successfully!")


In [ ]:

from pathlib import Path
import pandas as pd

project_dir = Path.cwd().parent

yield_df = pd.read_csv(
    project_dir / "data/raw/da_agriculture_yields.csv"
)

soil_df = pd.read_csv(
    project_dir / "data/external/Crop_recommendation.csv"
)

print("EXISTING YIELD DATASET:", yield_df.shape)
print("KAGGLE SOIL-WEATHER DATASET:", soil_df.shape)

print("\nYIELD DATASET COLUMNS:")
print(yield_df.columns.tolist())

print("\nKAGGLE DATASET COLUMNS:")
print(soil_df.columns.tolist())

print("\nEXISTING MODEL RESULTS:")
metrics_path = project_dir / "reports/week3_model_metrics.csv"

if metrics_path.exists():
    print(pd.read_csv(metrics_path).to_string(index=False))
else:
    print("Model metrics file not found.")

print("\nTARGET CROP COVERAGE IN KAGGLE:")
target_crops = ["rice", "soybean", "cotton", "sugarcane", "tomato"]

for crop in target_crops:
    count = (soil_df["label"].str.lower() == crop).sum()
    print(f"{crop}: {count}")


In [ ]:

import pandas as pd
from pathlib import Path

project_dir = Path.cwd().parent

enhanced_df = pd.read_csv(
    project_dir / "data/raw/agrimind_crop_disease_data.csv"
)

print("DATASET SHAPE:", enhanced_df.shape)

print("\nCOLUMNS:")
print(enhanced_df.columns.tolist())

print("\nTARGET CROP COUNTS:")
target_crops = ["Rice", "Soybean", "Cotton", "Sugarcane", "Tomato"]

print(
    enhanced_df[
        enhanced_df["crop"].isin(target_crops)
    ]["crop"].value_counts()
)

print("\nSAMPLE RECORDS:")
print(
    enhanced_df[
        enhanced_df["crop"].isin(target_crops)
    ].head().to_string(index=False)
)

print("\nMISSING VALUES:")
print(enhanced_df.isnull().sum())


In [ ]:

# STEP 4: Prepare enhanced crop yield dataset

import pandas as pd

target_crops = [
    "Rice", "Soybean", "Cotton",
    "Sugarcane", "Tomato"
]

enhanced_data = enhanced_df[
    enhanced_df["crop"].isin(target_crops)
].copy()

enhanced_data["date_recorded"] = pd.to_datetime(
    enhanced_data["date_recorded"]
)

enhanced_data["year"] = (
    enhanced_data["date_recorded"].dt.year
)

feature_columns = [
    "soil_ph",
    "temperature_c",
    "humidity_pct",
    "rainfall_mm",
    "nitrogen_kg_ha",
    "phosphorus_kg_ha",
    "potassium_kg_ha",
    "state",
    "season"
]

target_column = "yield_tons_ha"

print("ENHANCED DATASET READY")
print("Total records:", len(enhanced_data))

print("\nYEAR DISTRIBUTION:")
print(
    enhanced_data["year"]
    .value_counts()
    .sort_index()
)

print("\nCROP DISTRIBUTION:")
print(enhanced_data["crop"].value_counts())

print("\nFEATURES:")
print(feature_columns)

print("\nMISSING VALUES:")
print(
    enhanced_data[
        feature_columns + [target_column]
    ].isnull().sum()
)


In [ ]:

# STEP 5: Train enhanced models using soil and weather features

import numpy as np
import pandas as pd
import joblib

from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

numeric_features = [
    "soil_ph",
    "temperature_c",
    "humidity_pct",
    "rainfall_mm",
    "nitrogen_kg_ha",
    "phosphorus_kg_ha",
    "potassium_kg_ha"
]

categorical_features = ["state", "season"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

project_dir = Path.cwd().parent
model_dir = project_dir / "models" / "week3_enhanced"
report_dir = project_dir / "reports"

model_dir.mkdir(parents=True, exist_ok=True)
report_dir.mkdir(parents=True, exist_ok=True)

results = []

for crop in target_crops:

    crop_data = enhanced_data[
        enhanced_data["crop"] == crop
    ].copy()

    train = crop_data[crop_data["year"] == 2023]
    test = crop_data[crop_data["year"] == 2024]

    if train.empty or test.empty:
        print(f"Skipping {crop}: insufficient yearly data")
        continue

    X_train = train[feature_columns]
    y_train = train[target_column]

    X_test = test[feature_columns]
    y_test = test[target_column]

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                random_state=42,
                min_samples_leaf=2,
                n_jobs=-1
            )
        )
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    rmse = np.sqrt(
        mean_squared_error(y_test, predictions)
    )

    mape = (
        mean_absolute_percentage_error(
            y_test, predictions
        ) * 100
    )

    results.append({
        "crop": crop,
        "train_records": len(train),
        "test_records": len(test),
        "RMSE": round(rmse, 4),
        "MAPE_percent": round(mape, 2)
    })

    joblib.dump(
        pipeline,
        model_dir / f"{crop.lower()}_enhanced.joblib"
    )

    print(f"{crop} model trained successfully!")

enhanced_results = pd.DataFrame(results)

print("\nENHANCED MODEL RESULTS")
print(enhanced_results.to_string(index=False))

enhanced_results.to_csv(
    report_dir / "week3_enhanced_model_metrics.csv",
    index=False
)

print("\nEnhanced model metrics saved successfully!")


In [ ]:

# STEP 6: Fair Cotton feature comparison

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
import numpy as np
import pandas as pd

cotton = enhanced_data[
    enhanced_data["crop"] == "Cotton"
].copy()

train = cotton[cotton["year"] == 2023]
test = cotton[cotton["year"] == 2024]

basic_features = [
    "rainfall_mm", "state", "season"
]

enhanced_features = [
    "rainfall_mm",
    "temperature_c",
    "humidity_pct",
    "soil_ph",
    "nitrogen_kg_ha",
    "phosphorus_kg_ha",
    "potassium_kg_ha",
    "state",
    "season"
]

comparison = []

for name, features in [
    ("Basic", basic_features),
    ("Enhanced", enhanced_features)
]:
    numeric = [
        c for c in features
        if c not in ["state", "season"]
    ]

    preprocessor = ColumnTransformer([
        ("numeric", "passthrough", numeric),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            ["state", "season"]
        )
    ])

    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    pipeline.fit(train[features], train["yield_tons_ha"])

    pred = pipeline.predict(test[features])

    rmse = np.sqrt(
        mean_squared_error(test["yield_tons_ha"], pred)
    )

    mape = mean_absolute_percentage_error(
        test["yield_tons_ha"], pred
    ) * 100

    comparison.append({
        "model": name,
        "RMSE": round(rmse, 4),
        "MAPE_percent": round(mape, 2)
    })

cotton_comparison = pd.DataFrame(comparison)

print("COTTON FEATURE COMPARISON")
print(cotton_comparison.to_string(index=False))

cotton_comparison.to_csv(
    project_dir / "reports/week3_cotton_soil_comparison.csv",
    index=False
)

print("\nCotton comparison saved!")


In [ ]:

# STEP 7: Check whether valid lag features are possible

print("AVAILABLE YEARS:")
print(sorted(enhanced_data["year"].unique()))

print("\nSTATES PER CROP:")
print(
    enhanced_data.groupby("crop")["state"]
    .nunique()
)

print("\nRECORDS PER CROP AND YEAR:")
print(
    pd.crosstab(
        enhanced_data["crop"],
        enhanced_data["year"]
    )
)

print("\nDUPLICATE CROP-STATE-YEAR COMBINATIONS:")
duplicates = enhanced_data.duplicated(
    subset=["crop", "state", "year"],
    keep=False
)

print("Records sharing these keys:", duplicates.sum())

print("\nDISTRICT COLUMN AVAILABLE?")
print("district" in enhanced_data.columns)

print("\nSOIL TYPE COLUMN AVAILABLE?")
print("soil_type" in enhanced_data.columns)


In [ ]:

# STEP 8: Generate Week 3 checkpoint report

from pathlib import Path
import pandas as pd

project_dir = Path.cwd().parent
report_dir = project_dir / "reports"

baseline = pd.read_csv(
    report_dir / "week3_model_metrics.csv"
)

enhanced = pd.read_csv(
    report_dir / "week3_enhanced_model_metrics.csv"
)

cotton = pd.read_csv(
    report_dir / "week3_cotton_soil_comparison.csv"
)

report = f"""# KisanAI - Week 3 Checkpoint Report

## 1. Objective
Train separate crop-yield prediction models for Rice,
Soybean, Cotton, Sugarcane and Tomato.

Evaluate using RMSE and MAPE and investigate
additional soil and weather features.

## 2. Original Models
Dataset: da_agriculture_yields.csv

{baseline.to_markdown(index=False)}

## 3. Enhanced Models
Dataset: agrimind_crop_disease_data.csv

Training year: 2023
Testing year: 2024

Features:
- Soil pH
- Nitrogen, phosphorus and potassium
- Temperature and humidity
- Rainfall
- State and season

{enhanced.to_markdown(index=False)}

## 4. Cotton Feature Experiment

{cotton.to_markdown(index=False)}

The enhanced Cotton model achieved lower RMSE
and MAPE than the basic model on the same
2024 test records.

However, the percentage error remains high.

## 5. Limitations
- The two model groups use different datasets.
- Their results cannot establish a direct
  improvement over the original models.
- Soil pH is available, but soil type is not.
- District and farm identifiers are unavailable.
- Reliable previous-year yield lag features
  could not be constructed.
- Real-world generalisation is unverified.

## 6. Checkpoint Status
Completed:
- Five original crop-yield models
- Five enhanced crop-yield models
- RMSE and MAPE evaluation
- Cotton soil-and-weather feature experiment

Remaining:
- Genuine soil-type features
- Valid historical yield lag features
- Further improvements to Rice, Sugarcane
  and Cotton model accuracy
"""

output_path = report_dir / "week3_checkpoint_report.md"

output_path.write_text(
    report,
    encoding="utf-8"
)

print("Report saved successfully!")
print("Location:", output_path)


In [ ]:

from pathlib import Path

project_dir = Path.cwd().parent

external_dir = project_dir / "data" / "external"
external_dir.mkdir(parents=True, exist_ok=True)

print("External data folder ready:")
print(external_dir)

print("\nExisting files:")
for file in external_dir.iterdir():
    print(file.name)


In [ ]:

from pathlib import Path

external_dir = Path.cwd().parent / "data" / "external"

print("AVAILABLE CSV FILES:")
for file in external_dir.glob("*.csv"):
    print(file.name)


In [ ]:

# STEP 11: Inspect district-level agricultural dataset

import pandas as pd
from pathlib import Path

project_dir = Path.cwd().parent

district_path = (
    project_dir / "data" / "external"
    / "India Agriculture Crop Production.csv"
)

district_df = pd.read_csv(district_path)

print("DATASET SHAPE:")
print(district_df.shape)

print("\nCOLUMN NAMES:")
print(district_df.columns.tolist())

print("\nFIRST FIVE ROWS:")
print(district_df.head().to_string(index=False))

print("\nMISSING VALUES:")
print(district_df.isnull().sum())

print("\nSAMPLE CROP NAMES:")
crop_columns = [
    c for c in district_df.columns
    if "crop" in c.lower()
]

if crop_columns:
    crop_col = crop_columns[-1]
    print(district_df[crop_col].value_counts().head(30))
else:
    print("Crop column not identified automatically")


In [ ]:

# STEP 12: Clean district-level crop yield data

import pandas as pd

district_clean = district_df.copy()

# Standardize column names
district_clean.columns = (
    district_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

# Clean text columns
for col in ["state", "district", "crop", "season"]:
    district_clean[col] = (
        district_clean[col]
        .astype("string")
        .str.strip()
    )

# Normalize crop names
crop_mapping = {
    "Rice": "Rice",
    "Soyabean": "Soybean",
    "Soybean": "Soybean",
    "Cotton(lint)": "Cotton",
    "Sugarcane": "Sugarcane",
    "Tomato": "Tomato"
}

district_clean["crop"] = (
    district_clean["crop"].map(crop_mapping)
)

# Filter states and crops
target_states = [
    "Karnataka",
    "Maharashtra",
    "Telangana"
]

target_crops = [
    "Rice",
    "Soybean",
    "Cotton",
    "Sugarcane",
    "Tomato"
]

district_clean = district_clean[
    district_clean["state"].isin(target_states)
    & district_clean["crop"].isin(target_crops)
].copy()

# Extract start of agricultural year
district_clean["year_start"] = pd.to_numeric(
    district_clean["year"].str.extract(
        r"^(\d{4})", expand=False
    ),
    errors="coerce"
)

# Convert numeric columns
for col in ["area", "production", "yield"]:
    district_clean[col] = pd.to_numeric(
        district_clean[col],
        errors="coerce"
    )

# Keep usable records
district_clean = district_clean.dropna(
    subset=[
        "state", "district", "crop",
        "year_start", "area", "yield"
    ]
)

district_clean = district_clean[
    (district_clean["area"] > 0)
    & (district_clean["yield"] >= 0)
].copy()

print("CLEAN DATASET SHAPE:")
print(district_clean.shape)

print("\nCROP DISTRIBUTION:")
print(district_clean["crop"].value_counts())

print("\nSTATE DISTRIBUTION:")
print(district_clean["state"].value_counts())

print("\nYEAR RANGE:")
print(
    district_clean["year_start"].min(),
    "-",
    district_clean["year_start"].max()
)

print("\nNUMBER OF DISTRICTS:")
print(district_clean["district"].nunique())

print("\nYIELD UNITS:")
print(
    district_clean["production_units"]
    .value_counts()
)

print("\nAREA UNITS:")
print(
    district_clean["area_units"]
    .value_counts()
)


In [ ]:

# STEP 13: Validate district-year records

keys = ["state", "district", "crop", "year_start"]

print("DUPLICATE DISTRICT-CROP-YEAR GROUPS:")
group_sizes = district_clean.groupby(keys).size()

print("Groups with multiple records:",
      (group_sizes > 1).sum())

print("\nSAMPLE DUPLICATE GROUPS:")
print(
    group_sizes[group_sizes > 1]
    .head(10)
    .to_string()
)

print("\nPRODUCTION UNITS BY CROP:")
print(
    pd.crosstab(
        district_clean["crop"],
        district_clean["production_units"]
    )
)

print("\nSEASONS BY CROP:")
print(
    pd.crosstab(
        district_clean["crop"],
        district_clean["season"]
    )
)

print("\nTOMATO IN ORIGINAL DATASET:")
tomato_rows = district_df[
    district_df["Crop"]
    .astype("string")
    .str.strip()
    .str.lower()
    .eq("tomato")
]

print("Total Tomato records:", len(tomato_rows))

print("\nSAMPLE TOMATO STATES:")
print(tomato_rows["State"].value_counts().head(10))


In [ ]:

# STEP 14: Create district-level annual yield data

import pandas as pd
from pathlib import Path

keys = ["state", "district", "crop", "year_start"]

# Remove rows without valid production or area
annual_source = district_clean.dropna(
    subset=["production", "area", "production_units"]
).copy()

annual_source = annual_source[
    (annual_source["area"] > 0)
    & (annual_source["production"] >= 0)
].copy()

# Check for mixed production units within a group
unit_counts = annual_source.groupby(keys)[
    "production_units"
].nunique()

assert unit_counts.max() == 1, (
    "Mixed production units found within a district-crop-year"
)

# Combine seasonal production and area
annual_df = (
    annual_source.groupby(
        keys + ["production_units"],
        as_index=False
    )
    .agg(
        annual_area=("area", "sum"),
        annual_production=("production", "sum"),
        season_records=("season", "size")
    )
)

# Annual yield in the source production unit per hectare
annual_df["annual_yield"] = (
    annual_df["annual_production"]
    / annual_df["annual_area"]
)

# Sort records chronologically
annual_df = annual_df.sort_values(keys).reset_index(drop=True)

group_cols = ["state", "district", "crop"]

# Previous observed year and yield
annual_df["previous_year"] = (
    annual_df.groupby(group_cols)["year_start"].shift(1)
)

annual_df["previous_yield"] = (
    annual_df.groupby(group_cols)["annual_yield"].shift(1)
)

# Keep lag only when previous observation is exactly one year earlier
valid_lag = (
    annual_df["year_start"] - annual_df["previous_year"] == 1
)

annual_df["yield_lag_1"] = (
    annual_df["previous_yield"].where(valid_lag)
)

# Remove helper columns
annual_df = annual_df.drop(
    columns=["previous_year", "previous_yield"]
)

# Save dataset
project_dir = Path.cwd().parent
output_path = (
    project_dir / "data" / "processed"
    / "district_annual_yield_lags.csv"
)

output_path.parent.mkdir(parents=True, exist_ok=True)
annual_df.to_csv(output_path, index=False)

print("ANNUAL DATASET:", annual_df.shape)

print("\nRECORDS WITH VALID PREVIOUS-YEAR YIELD:")
print(annual_df["yield_lag_1"].notna().sum())

print("\nVALID LAGS BY CROP:")
print(
    annual_df.groupby("crop")["yield_lag_1"]
    .apply(lambda x: x.notna().sum())
)

print("\nANNUAL RECORDS BY CROP:")
print(annual_df["crop"].value_counts())

print("\nSAMPLE:")
print(
    annual_df[
        annual_df["yield_lag_1"].notna()
    ].head(10).to_string(index=False)
)

print("\nSaved to:", output_path)


In [ ]:

# STEP 15: Historical yield prediction with lag features

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

historical = annual_df.dropna(
    subset=["yield_lag_1"]
).copy()

numeric_cols = [
    "year_start",
    "yield_lag_1"
]

categorical_cols = [
    "state",
    "district"
]

feature_cols = numeric_cols + categorical_cols

historical_results = []

model_dir = project_dir / "models" / "week3_historical"
model_dir.mkdir(parents=True, exist_ok=True)

for crop in ["Rice", "Soybean", "Cotton", "Sugarcane"]:

    crop_df = historical[
        historical["crop"] == crop
    ].copy()

    # Train on earlier years, test on later years
    train = crop_df[crop_df["year_start"] <= 2015]
    test = crop_df[crop_df["year_start"] > 2015]

    if train.empty or test.empty:
        print(f"Skipping {crop}: insufficient data")
        continue

    preprocessor = ColumnTransformer([
        ("num", "passthrough", numeric_cols),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    model.fit(
        train[feature_cols],
        train["annual_yield"]
    )

    predictions = model.predict(
        test[feature_cols]
    )

    rmse = np.sqrt(
        mean_squared_error(
            test["annual_yield"],
            predictions
        )
    )

    mape = mean_absolute_percentage_error(
        test["annual_yield"],
        predictions
    ) * 100

    historical_results.append({
        "crop": crop,
        "train_records": len(train),
        "test_records": len(test),
        "RMSE": round(rmse, 4),
        "MAPE_percent": round(mape, 2),
        "yield_unit": (
            "Bales/ha" if crop == "Cotton"
            else "Tonnes/ha"
        )
    })

    joblib.dump(
        model,
        model_dir / f"{crop.lower()}_historical.joblib"
    )

    print(f"{crop} historical model trained!")

historical_metrics = pd.DataFrame(
    historical_results
)

print("\nHISTORICAL MODEL RESULTS")
print(
    historical_metrics.to_string(index=False)
)

historical_metrics.to_csv(
    project_dir / "reports"
    / "week3_historical_model_metrics.csv",
    index=False
)

print("\nHistorical model metrics saved!")


In [ ]:

# STEP 16: Inspect district coverage for soil mapping

district_summary = (
    annual_df.groupby(["state", "district"])
    .agg(
        records=("crop", "size"),
        crops=("crop", "nunique"),
        first_year=("year_start", "min"),
        last_year=("year_start", "max")
    )
    .reset_index()
)

print("DISTRICTS PER STATE:")
print(
    district_summary.groupby("state")["district"]
    .nunique()
)

print("\nSAMPLE DISTRICTS:")
for state in ["Karnataka", "Maharashtra", "Telangana"]:
    names = (
        district_summary.loc[
            district_summary["state"] == state,
            "district"
        ]
        .sort_values()
        .tolist()
    )

    print(f"\n{state} ({len(names)} districts):")
    print(", ".join(names))

print("\nTOTAL DISTRICT-STATE COMBINATIONS:")
print(len(district_summary))


In [ ]:

# STEP 17: Standardise district names

import pandas as pd

district_aliases = {
    "BAGALKOTE": "BAGALKOT",
    "BELGAUM": "BELAGAVI",
    "BELLARY": "BALLARI",
    "BANGALORE RURAL": "BENGALURU RURAL",
    "BIJAPUR": "VIJAYAPURA",
    "CHAMARAJANAGARA": "CHAMARAJANAGAR",
    "CHIKBALLAPUR": "CHIKKABALLAPURA",
    "CHIKMAGALUR": "CHIKKAMAGALURU",
    "DAKSHIN KANNAD": "DAKSHINA KANNADA",
    "GULBARGA": "KALABURAGI",
    "MYSORE": "MYSURU",
    "SHIMOGA": "SHIVAMOGGA",
    "TUMKUR": "TUMAKURU",
    "UTTAR KANNAD": "UTTARA KANNADA",
    "YADGIR": "YADAGIRI",
    "MAHBUBNAGAR": "MAHABUBNAGAR",
    "RANGAREDDI": "RANGAREDDY"
}

district_standardized = annual_df.copy()

district_standardized["district_original"] = (
    district_standardized["district"]
)

district_standardized["district"] = (
    district_standardized["district"]
    .replace(district_aliases)
)

# Flag suspicious state-district combinations
invalid_telangana = [
    "DADRA AND NAGAR HAVELI",
    "DAMAN"
]

suspicious = district_standardized[
    (district_standardized["state"] == "Telangana")
    & district_standardized["district"].isin(invalid_telangana)
]

print("SUSPICIOUS RECORDS:")
print(
    suspicious[
        ["state", "district", "crop", "year_start"]
    ].to_string(index=False)
)

print("\nDISTRICTS AFTER STANDARDISATION:")
print(
    district_standardized.groupby("state")["district"]
    .nunique()
)

print("\nDUPLICATE STANDARDISED KEYS:")
print(
    district_standardized.duplicated(
        ["state", "district", "crop", "year_start"],
        keep=False
    ).sum()
)


In [ ]:

# STEP 18: Clean suspicious records and recalculate lags

import pandas as pd

clean_history = district_standardized.copy()

# Remove known invalid Telangana district records
invalid = (
    (clean_history["state"] == "Telangana")
    & clean_history["district"].isin([
        "DADRA AND NAGAR HAVELI",
        "DAMAN"
    ])
)

clean_history = clean_history.loc[~invalid].copy()

# Confirm unique district-crop-year keys
keys = ["state", "district", "crop", "year_start"]

assert not clean_history.duplicated(keys).any(), (
    "Duplicate district-crop-year records found"
)

# Recalculate lag features after standardisation
clean_history = clean_history.sort_values(
    keys
).reset_index(drop=True)

groups = ["state", "district", "crop"]

previous_year = (
    clean_history.groupby(groups)["year_start"].shift(1)
)

previous_yield = (
    clean_history.groupby(groups)["annual_yield"].shift(1)
)

clean_history["yield_lag_1"] = previous_yield.where(
    clean_history["year_start"] - previous_year == 1
)

# Save a new version without overwriting original data
output_path = (
    project_dir / "data" / "processed"
    / "district_annual_yield_lags_clean.csv"
)

clean_history.to_csv(output_path, index=False)

print("CLEAN HISTORICAL DATASET:", clean_history.shape)

print("\nVALID LAGS:")
print(clean_history["yield_lag_1"].notna().sum())

print("\nDISTRICTS PER STATE:")
print(
    clean_history.groupby("state")["district"].nunique()
)

print("\nLAGS BY CROP:")
print(
    clean_history.groupby("crop")["yield_lag_1"]
    .apply(lambda x: x.notna().sum())
)

print("\nSAVED TO:")
print(output_path)


In [ ]:

# STEP 19: Export districts for soil-data matching

district_lookup = (
    clean_history[["state", "district"]]
    .drop_duplicates()
    .sort_values(["state", "district"])
    .reset_index(drop=True)
)

district_lookup["soil_type"] = pd.NA
district_lookup["soil_source"] = pd.NA

output_path = (
    project_dir / "data" / "external"
    / "district_soil_lookup.csv"
)

district_lookup.to_csv(output_path, index=False)

print("District lookup created!")
print("Districts:", len(district_lookup))
print("File:", output_path)

print("\nFIRST 10 ROWS:")
print(district_lookup.head(10).to_string(index=False))


In [ ]:

# STEP 21: Find soil-related features in existing datasets

import pandas as pd
from pathlib import Path

keywords = [
    "soil", "ph", "nitrogen",
    "phosphorus", "potassium",
    "texture", "district"
]

for folder_name in ["raw", "external", "processed"]:

    folder = project_dir / "data" / folder_name

    for file in sorted(folder.glob("*.csv")):

        try:
            df_check = pd.read_csv(file, nrows=5)

            matches = [
                col for col in df_check.columns
                if any(
                    word in col.lower()
                    for word in keywords
                )
            ]

            if matches:
                print(f"\nFILE: {file.name}")
                print("MATCHING COLUMNS:", matches)

        except Exception as e:
            print(f"Could not read {file.name}: {e}")

print("\nSoil feature inspection completed!")


In [ ]:

# STEP 22: Train models using cleaned district history

import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_percentage_error
)

data = clean_history.dropna(
    subset=["yield_lag_1", "annual_yield"]
).copy()

features = [
    "year_start",
    "yield_lag_1",
    "state",
    "district"
]

results = []

model_dir = project_dir / "models" / "week3_final"
model_dir.mkdir(parents=True, exist_ok=True)

for crop in ["Rice", "Soybean", "Cotton", "Sugarcane"]:

    crop_data = data[data["crop"] == crop].copy()

    train = crop_data[crop_data["year_start"] <= 2015]
    test = crop_data[crop_data["year_start"] > 2015]

    if train.empty or test.empty:
        print(f"Skipping {crop}: insufficient data")
        continue

    preprocessor = ColumnTransformer([
        (
            "numeric",
            "passthrough",
            ["year_start", "yield_lag_1"]
        ),
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            ["state", "district"]
        )
    ])

    model = Pipeline([
        ("preprocessor", preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=300,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            )
        )
    ])

    model.fit(
        train[features],
        train["annual_yield"]
    )

    predictions = model.predict(test[features])

    rmse = np.sqrt(
        mean_squared_error(
            test["annual_yield"], predictions
        )
    )

    mape = mean_absolute_percentage_error(
        test["annual_yield"], predictions
    ) * 100

    results.append({
        "Crop": crop,
        "Train": len(train),
        "Test": len(test),
        "RMSE": round(rmse, 4),
        "MAPE (%)": round(mape, 2),
        "Unit": (
            "Bales/ha" if crop == "Cotton"
            else "Tonnes/ha"
        )
    })

    joblib.dump(
        model,
        model_dir / f"{crop.lower()}_model.joblib"
    )

    print(f"{crop} model trained successfully!")

final_metrics = pd.DataFrame(results)

print("\nCLEANED HISTORICAL MODEL RESULTS:")
print(final_metrics.to_string(index=False))

report_path = (
    project_dir / "reports"
    / "week3_cleaned_historical_metrics.csv"
)

final_metrics.to_csv(report_path, index=False)

print("\nSaved:", report_path)


In [ ]:

# STEP 23: Compare ML models against previous-year baseline

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_percentage_error
)

comparison = []

features = [
    "year_start",
    "yield_lag_1",
    "state",
    "district"
]

for crop in ["Rice", "Soybean", "Cotton", "Sugarcane"]:

    crop_data = clean_history[
        (clean_history["crop"] == crop)
        & clean_history["yield_lag_1"].notna()
        & clean_history["annual_yield"].notna()
    ].copy()

    test = crop_data[
        crop_data["year_start"] > 2015
    ].copy()

    model_path = (
        project_dir / "models" / "week3_final"
        / f"{crop.lower()}_model.joblib"
    )

    model = joblib.load(model_path)

    actual = test["annual_yield"].to_numpy()

    ml_pred = model.predict(test[features])

    baseline_pred = test["yield_lag_1"].to_numpy()

    # Avoid undefined percentage errors when actual yield is zero
    nonzero = actual != 0

    def calculate_mape(pred):
        if not nonzero.any():
            return np.nan
        return (
            mean_absolute_percentage_error(
                actual[nonzero],
                pred[nonzero]
            ) * 100
        )

    ml_rmse = np.sqrt(
        mean_squared_error(actual, ml_pred)
    )

    baseline_rmse = np.sqrt(
        mean_squared_error(actual, baseline_pred)
    )

    comparison.append({
        "Crop": crop,
        "ML_RMSE": round(ml_rmse, 4),
        "Baseline_RMSE": round(baseline_rmse, 4),
        "ML_MAPE": round(calculate_mape(ml_pred), 2),
        "Baseline_MAPE": round(
            calculate_mape(baseline_pred), 2
        ),
        "RMSE_Winner": (
            "ML" if ml_rmse < baseline_rmse
            else "Baseline"
        )
    })

baseline_comparison = pd.DataFrame(comparison)

print("\nMODEL VS PREVIOUS-YEAR BASELINE")
print(baseline_comparison.to_string(index=False))

output_path = (
    project_dir / "reports"
    / "week3_baseline_comparison.csv"
)

baseline_comparison.to_csv(output_path, index=False)

print("\nSaved:", output_path)


In [ ]:

# STEP 24: Test Gradient Boosting against existing models

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error

features = [
    "year_start",
    "yield_lag_1",
    "state",
    "district"
]

results = []

for crop in ["Rice", "Soybean", "Sugarcane"]:

    crop_df = clean_history[
        (clean_history["crop"] == crop)
        & clean_history["yield_lag_1"].notna()
    ].copy()

    train = crop_df[crop_df["year_start"] <= 2015]
    test = crop_df[crop_df["year_start"] > 2015]

    preprocessing = ColumnTransformer([
        (
            "numeric",
            "passthrough",
            ["year_start", "yield_lag_1"]
        ),
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            ["state", "district"]
        )
    ])

    model = Pipeline([
        ("preprocessing", preprocessing),
        (
            "regressor",
            HistGradientBoostingRegressor(
                max_iter=200,
                learning_rate=0.05,
                max_leaf_nodes=15,
                l2_regularization=1.0,
                random_state=42
            )
        )
    ])

    model.fit(
        train[features],
        train["annual_yield"]
    )

    predictions = model.predict(test[features])

    rmse = np.sqrt(
        mean_squared_error(
            test["annual_yield"],
            predictions
        )
    )

    baseline_rmse = np.sqrt(
        mean_squared_error(
            test["annual_yield"],
            test["yield_lag_1"]
        )
    )

    results.append({
        "Crop": crop,
        "GradientBoosting_RMSE": round(rmse, 4),
        "Baseline_RMSE": round(baseline_rmse, 4),
        "Better": (
            "Gradient Boosting"
            if rmse < baseline_rmse
            else "Baseline"
        )
    })

gb_results = pd.DataFrame(results)

print("\nGRADIENT BOOSTING RESULTS")
print(gb_results.to_string(index=False))


In [ ]:

# STEP 25: Generate Week 3 checkpoint report

from pathlib import Path

report = """# KisanAI AgriTech — Week 3 Checkpoint Report

## Objective
Train crop-specific yield prediction models and evaluate
their performance using RMSE and MAPE.

## Dataset Preparation
- Historical crop production records from Karnataka,
  Maharashtra and Telangana.
- 4,916 cleaned district-crop-year records.
- 4,415 records with valid previous-year yield features.
- 98 state-district combinations after cleaning.
- Historical crops: Rice, Soybean, Cotton and Sugarcane.
- Tomato evaluated separately using another dataset.

## Historical Model Features
- State
- District
- Year
- Previous-year yield

Historical models do not yet combine weather and soil type.

## Cleaned Random Forest Results

| Crop | RMSE | MAPE | Unit |
|---|---:|---:|---|
| Rice | 0.5816 | 23.04% | Tonnes/ha |
| Soybean | 0.5849 | 38.14% | Tonnes/ha |
| Cotton | 1.3449 | 41.75% | Bales/ha |
| Sugarcane | 21.3894 | 25.59% | Tonnes/ha |

## Baseline Comparison

| Crop | Random Forest RMSE | Baseline RMSE |
|---|---:|---:|
| Rice | 0.5816 | 0.5521 |
| Soybean | 0.5849 | 0.4872 |
| Cotton | 1.3449 | 1.5686 |
| Sugarcane | 21.3894 | 18.1121 |

Cotton Random Forest outperformed the previous-year
baseline by approximately 14.26% in RMSE.

## Gradient Boosting Results

| Crop | Gradient Boosting RMSE |
|---|---:|
| Rice | 0.6108 |
| Soybean | 0.5285 |
| Sugarcane | 19.6701 |

Gradient Boosting did not outperform the previous-year
baseline on the tested crops.

## Soil Feature Investigation
The AgriMind dataset includes:
- Soil pH
- Nitrogen
- Phosphorus
- Potassium

An earlier experiment tested these features for Cotton,
but the dataset lacks district-level identifiers needed
to merge them reliably with historical crop records.

Categorical soil type remains unavailable.

## Limitations
1. Historical data does not contain weather measurements.
2. Verified district-level soil-type data is unavailable.
3. Historical Tomato records are unavailable.
4. Cotton yields are in bales per hectare, while other
   historical crops use tonnes per hectare.
5. Most ML models did not outperform the baseline.
6. Historical annual yield aggregation and forecasting
   assumptions require further validation.

## Next Steps
- Obtain verified geographically matched soil data.
- Integrate historical weather data where possible.
- Improve Cotton predictions and reduce MAPE.
- Evaluate additional forecasting methods.
- Validate models on future-year holdout data.

## Conclusion
Four historical crop-specific Random Forest models were
trained and evaluated. Cotton showed improvement over
the previous-year baseline. Other crops require further
development. The combined weather, lag and categorical
soil-type modelling requirement remains incomplete.
"""

report_path = (
    project_dir / "reports" /
    "week3_final_checkpoint_report.md"
)

report_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

report_path.write_text(
    report,
    encoding="utf-8"
)

print("Week 3 checkpoint report created!")
print("Saved to:", report_path)


In [ ]:

# STEP 26: Verify Week 3 submission files

from pathlib import Path

required_files = [
    "notebooks/02_crop_yield_models.ipynb",
    "data/processed/district_annual_yield_lags_clean.csv",
    "reports/week3_cleaned_historical_metrics.csv",
    "reports/week3_baseline_comparison.csv",
    "reports/week3_final_checkpoint_report.md",
    "models/week3_final/rice_model.joblib",
    "models/week3_final/soybean_model.joblib",
    "models/week3_final/cotton_model.joblib",
    "models/week3_final/sugarcane_model.joblib"
]

print("WEEK 3 SUBMISSION FILE CHECK\n")

missing = []

for relative_path in required_files:
    file_path = project_dir / relative_path

    if file_path.is_file():
        print("OK     ", relative_path)
    else:
        print("MISSING", relative_path)
        missing.append(relative_path)

print("\nTotal files checked:", len(required_files))
print("Missing files:", len(missing))

if not missing:
    print("\nAll required files are present!")
else:
    print("\nResolve missing files before submission.")
